In [ ]:
from collections import defaultdict

spielbrett = {}  # dictionary

SPALTEN = 7
ZEILEN = 6
ZELLEN = SPALTEN * ZEILEN
RICHTUNGEN = [(-1, -1), (0, -1), (1, -1), (-1, 0),
                  (1, 0), (-1, 1), (0, 1), (1, 1)]
quads_indices = defaultdict(list)


def stein_loeschen(pos, spieler):
  del spielbrett[pos]
  for i in quads_indices[pos]:
    quads[i][spieler] -=1

def bewerten():
  score = 0
  for pos in spielbrett:
    for i in quads_indices[pos]:
      gelbe, rote = quads[i]
      if gelbe > 0 and rote > 0: continue
      score += rote*10
      score -= gelbe*10
  return score

def zug_liste():
  zuege = []
  for spalte in range(SPALTEN):
    if not spalte_ist_gültig(spalte): continue
    zeile = finde_tiefste_zeile(spalte)
    zuege.append((spalte,zeile))
  return zuege

def spieler_mensch(spieler):
  while True:
    spalte = int(input('Ihr Zug (Spalte 0-6): '))
    if spalte_ist_gueltig(spalte):
      break
  zeile = finde_tiefste_zeile(spalte)
  win = stein_setzen((spalte,zeile), spieler)
  return win

def spieler_computer(spieler):
  bewertete_zuege = []
  for zug in zug_liste():
    win = stein_setzen(zug, spieler)
    score = minimax(7, -999999, 999999, spieler, win)
    stein_loeschen(zug, spieler)
    bewertete_zuege.append((score,zug))
  bewertete_zuege.sort(reverse=spieler)
  score, bester_zug = bewertete_zuege[0]
  win = stein_setzen(bester_zug, spieler)
  print(bewertete_zuege)
  print(f'Spieler {1 if spieler else 2} setzt {bester_zug} mit der Bewertung {score}')
  return win

# Alpha-Beta-Suche
def minimax(tiefe, alpha, beta, spieler, win):
  if win:
    return 99999+tiefe if spieler else -99999-tiefe
  if tiefe == 0 or len(spielbrett) == ZELLEN:
    return bewerten()
  spieler = not spieler
  value = -999999 if spieler else 999999
  for zug in zug_liste():
    win = stein_setzen(zug, spieler)
    score = minimax(tiefe-1, alpha, beta, spieler, win)
    stein_loeschen(zug, spieler)
    if spieler:
      value = max(value, score)
      alpha = max(value, alpha)
    else:
      value = min(value, score)
      beta = min(value, beta)
    if alpha >= beta:
      break
  return value

quads = quads_bestimmen()


spieler = True
while True:
  print_spielbrett()
  if spieler:
    win = spieler_mensch(spieler)
  else:
    win = spieler_computer(spieler)
  if win:
    print_spielbrett()
    print('GEWONNEN!')
    break
  spieler = not spieler
